# 42173 Advanced Natural Language Processing
## Component: Generator evaluation on PubHealth (no retrieval vs RAG vs CRAG)

This notebook evaluates the group's generator (`src/crag/generator.py`, Qwen2.5-Instruct) on PubHealth health claims
in three settings. The generator, prompt, claims and decoding are the same in every setting, so any difference in
accuracy comes from the knowledge the generator is given.

| Setting | Knowledge given to the generator |
|---|---|
| No retrieval | none, the model answers from what it learnt in pre-training |
| Standard RAG | the top-k MedRAG Textbooks passages from Contriever |
| CRAG | the same passages after the retrieval evaluator, action trigger, knowledge refinement and web search |

It reuses the group's code unchanged: the `MODELS` table and index path from `scripts/evaluate.py`,
`load_pubhealth`, `run_rag`, `correct_knowledge` and `predict_label`. The results are therefore comparable with
`results/rag_pubhealth_n200_3b.csv` and `results/crag_pubhealth_n200_0.5b.csv`.

Results are saved after every claim, so a run that stops part-way can be resumed by running the notebook again.

## 1. Setup

In [ ]:
# On Google Colab, uncomment these lines to get the repo (with its Git LFS model files) and the requirements.
# !git lfs install
# !git clone https://github.com/hulemihi/42173-ANLP-CRAG.git
# %cd 42173-ANLP-CRAG/notebooks
# !pip -q install -r ../requirements.txt

In [ ]:
import importlib.util
import json
import math
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# The repo root is the folder that contains src/crag (this notebook lives in notebooks/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "crag").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from crag.data import load_pubhealth, load_textbooks_corpus
from crag.evaluator import load_evaluator
from crag.generator import load_generator, predict_label
from crag.pipeline import check_crag_ready, correct_knowledge, run_rag
from crag.retriever import build_index, retrieve

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Repo root:", ROOT)
print("Using device:", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")

In [ ]:
# Reuse the model table and index path from scripts/evaluate.py, so this notebook and the script stay in sync
spec = importlib.util.spec_from_file_location("evaluate_script", ROOT / "scripts" / "evaluate.py")
evaluate_script = importlib.util.module_from_spec(spec)
spec.loader.exec_module(evaluate_script)

MODELS = evaluate_script.MODELS
INDEX_PATH = evaluate_script.INDEX_PATH
MODELS

## 2. Settings

`MODEL_KEY` picks the generator from `MODELS`. The 3B model is the default, as in `scripts/evaluate.py`.
It fits on a 16 GB GPU in bf16. Without a GPU it still runs: on a 16-thread laptop CPU in float32, one claim
takes about 40 seconds for all three settings, so the full 200 claims take about two hours.

In [ ]:
MODEL_KEY = "3b"        # "0.5b", "3b" or "7b" (7b is loaded in 4-bit and needs a GPU)
N_CLAIMS = 200          # same fixed, stratified PubHealth subset as scripts/evaluate.py
K = 5                   # passages retrieved per claim
N_EXAMPLES = 3          # claims shown with a full RAG and CRAG response at the end
CPU_FLOAT32 = True      # CPU only: run the bf16 weights in float32, about 4x faster on CPUs without bf16 support
                        # but needs twice the memory (about 13 GB for 3B). Ignored on a GPU.
RESUME = True           # skip claims that are already in the results file

In [ ]:
RESULTS_PATH = ROOT / "results" / f"crag_pubhealth_n{N_CLAIMS}_{MODEL_KEY}.csv"
RESULTS_PATH.parent.mkdir(exist_ok=True)

print("Generator:", MODELS[MODEL_KEY])
print("Results file:", RESULTS_PATH.relative_to(ROOT))
print("Web search:", "Tavily + PubMed" if os.getenv("TAVILY_API_KEY") else "PubMed only (no TAVILY_API_KEY in .env)")

## 3. Load the PubHealth claims

In [ ]:
claims = load_pubhealth(n=N_CLAIMS)
claims.insert(0, "id", range(len(claims)))

majority = claims.label.value_counts(normalize=True).max()
print(f"{len(claims)} claims: {claims.label.value_counts().to_dict()}")
print(f"Always answering the majority label gives {majority:.1%} accuracy")
claims[["id", "claim", "label"]].head()

## 4. Load the retriever index, the generator and the retrieval evaluator

In [ ]:
t0 = time.time()
index = build_index(load_textbooks_corpus(), cache_path=INDEX_PATH)
print(f"Contriever index: {len(index.passages):,} passages, ready in {time.time() - t0:.0f}s")

t0 = time.time()
model, tokenizer = load_generator(**MODELS[MODEL_KEY])
if DEVICE == "cpu" and CPU_FLOAT32:
    model = model.float()
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Generator: {MODELS[MODEL_KEY]['model_name']} ({n_params:.2f}B parameters, "
      f"{next(model.parameters()).dtype}), ready in {time.time() - t0:.0f}s")

eval_model, eval_tokenizer = load_evaluator()
problems = check_crag_ready()
print("CRAG stages ready" if not problems else "CRAG problems:\n" + "\n".join(problems))

## 5. Run the three settings

For each claim, the label is predicted three times with `explain=False`, exactly as in `scripts/evaluate.py`
(one forward pass, the label with the highest next-token probability):

1. **No retrieval**: `predict_label` with no passages.
2. **Standard RAG**: `run_rag`, the top-k textbook passages.
3. **CRAG**: the same steps as `pipeline.run_crag` (retrieve, `correct_knowledge`, `predict_label`), written out here
   so the evaluator scores and the size of the corrected knowledge can be saved as well.

In [ ]:
WEB_PREFIXES = ("[PubMed]", "[Web]")


def evaluate_claim(claim):
    row = {}

    t = time.time()
    none = predict_label(model, tokenizer, claim, [], explain=False)
    row.update(no_retrieval=none.label, no_retrieval_p_true=none.probs["true"],
               seconds_no_retrieval=time.time() - t)

    t = time.time()
    rag = run_rag(claim, index, model, tokenizer, k=K, explain=False)
    row.update(rag=rag.label, rag_p_true=rag.probs["true"], seconds_rag=time.time() - t)

    t = time.time()
    passages = retrieve(index, claim, k=K)
    knowledge, action, scores = correct_knowledge(claim, passages, eval_model, eval_tokenizer,
                                                  gen_model=model, gen_tokenizer=tokenizer)
    crag = predict_label(model, tokenizer, claim, knowledge, explain=False)
    row.update(crag=crag.label, crag_p_true=crag.probs["true"], crag_action=action,
               crag_scores=json.dumps([round(float(s), 4) for s in scores]),
               crag_max_score=max(scores),
               crag_knowledge_items=len(knowledge),
               crag_web_items=sum(k.startswith(WEB_PREFIXES) for k in knowledge),
               crag_knowledge_words=sum(len(k.split()) for k in knowledge),
               seconds_crag=time.time() - t)
    return row

In [ ]:
rows = []
if RESUME and RESULTS_PATH.exists():
    saved = pd.read_csv(RESULTS_PATH)
    if "id" in saved.columns:
        rows = saved[saved.id.isin(claims.id)].to_dict("records")
done = {r["id"] for r in rows}
print(f"Resuming with {len(done)} claims already done" if done else "Starting a new run")

start = time.time()
for r in tqdm(claims.itertuples(), total=len(claims), desc="Claims"):
    if r.id in done:
        continue
    rows.append({"id": r.id, "claim": r.claim, "gold": r.label, **evaluate_claim(r.claim)})
    pd.DataFrame(rows).sort_values("id").to_csv(RESULTS_PATH, index=False)

results = pd.DataFrame(rows).sort_values("id").reset_index(drop=True)
print(f"{len(results)} claims in {RESULTS_PATH.name} ({time.time() - start:.0f}s in this session)")
results.head()

## 6. Results

In [ ]:
SETTINGS = {"no_retrieval": "No retrieval", "rag": "Standard RAG", "crag": "CRAG"}
LABELS = ("true", "false")


def macro_f1(gold, pred):
    f1s = []
    for label in LABELS:
        tp = ((pred == label) & (gold == label)).sum()
        fp = ((pred == label) & (gold != label)).sum()
        fn = ((pred != label) & (gold == label)).sum()
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return float(np.mean(f1s))


for col in SETTINGS:
    results[f"{col}_correct"] = results[col] == results.gold

summary = pd.DataFrame([{
    "setting": name,
    "accuracy": results[f"{col}_correct"].mean(),
    "macro_f1": macro_f1(results.gold, results[col]),
    "predicted_true": int((results[col] == "true").sum()),
    "predicted_false": int((results[col] == "false").sum()),
    "mean_p_true": results[f"{col}_p_true"].mean(),
    "seconds_per_claim": results[f"seconds_{col}"].mean(),
} for col, name in SETTINGS.items()]).set_index("setting")

always_true = (results.gold == "true").mean()
print(f"{MODELS[MODEL_KEY]['model_name']} on {len(results)} PubHealth claims "
      f"({results.gold.value_counts().to_dict()}). Always 'true' accuracy = {always_true:.3f}")
summary.round(3)

In [ ]:
colours = ["#64748B", "#D97706", "#4F46E5"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(summary.index, summary.accuracy, color=colours)
axes[0].axhline(always_true, ls="--", color="black", lw=1, label=f"always 'true' ({always_true:.1%})")
for i, v in enumerate(summary.accuracy):
    axes[0].text(i, v + 0.01, f"{v:.1%}", ha="center")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Accuracy")
axes[0].set_title(f"PubHealth accuracy, Qwen2.5-{MODEL_KEY.upper()} (n={len(results)})")
axes[0].legend(loc="upper right")

bins = np.linspace(0, 1, 21)
for (col, name), colour in zip(SETTINGS.items(), colours):
    axes[1].hist(results[f"{col}_p_true"], bins=bins, alpha=0.55, color=colour, label=name)
axes[1].axvline(0.5, ls="--", color="black", lw=1)
axes[1].set_xlabel("P(true) from the generator")
axes[1].set_ylabel("Claims")
axes[1].set_title("How confident the generator is that a claim is true")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices (rows = gold label, columns = predicted label)
for col, name in SETTINGS.items():
    print(name)
    print(pd.crosstab(results.gold, results[col]).reindex(index=LABELS, columns=LABELS, fill_value=0), "\n")

### Accuracy by CRAG action

Correct keeps the refined textbook knowledge, Incorrect replaces it with web search results, and Ambiguous uses both.
Comparing the three settings on the same claims shows where CRAG's correction helps.

In [ ]:
ACTIONS = ["correct", "ambiguous", "incorrect"]
by_action = (results.groupby("crag_action")
             .agg(claims=("id", "size"),
                  no_retrieval=("no_retrieval_correct", "mean"),
                  rag=("rag_correct", "mean"),
                  crag=("crag_correct", "mean"),
                  mean_max_score=("crag_max_score", "mean"),
                  mean_web_items=("crag_web_items", "mean"))
             .reindex([a for a in ACTIONS if a in set(results.crag_action)]))
by_action.round(3)

In [ ]:
ax = by_action[["no_retrieval", "rag", "crag"]].plot.bar(color=colours, figsize=(8, 4), rot=0)
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_xlabel("CRAG action")
ax.set_title("Accuracy by CRAG action")
ax.set_xticklabels([f"{a}\n(n={n})" for a, n in zip(by_action.index, by_action.claims)])
ax.legend(list(SETTINGS.values()))
plt.tight_layout()
plt.show()

### Paired comparison (exact McNemar test)

All three settings answer the same claims, so they are compared on the claims where they disagree.
A p-value below 0.05 means the difference is unlikely to be chance on these claims.

In [ ]:
def mcnemar(a_correct, b_correct):
    """Exact two-sided McNemar test. Returns (only A right, only B right, p-value)."""
    only_a = int((a_correct & ~b_correct).sum())
    only_b = int((~a_correct & b_correct).sum())
    n = only_a + only_b
    if n == 0:
        return only_a, only_b, 1.0
    tail = sum(math.comb(n, i) for i in range(min(only_a, only_b) + 1)) / 2 ** n
    return only_a, only_b, min(1.0, 2 * tail)


pairs = [("rag", "no_retrieval"), ("crag", "no_retrieval"), ("crag", "rag")]
pd.DataFrame([
    dict(zip(["comparison", "only first right", "only second right", "p_value"],
             [f"{SETTINGS[a]} vs {SETTINGS[b]}",
              *mcnemar(results[f"{a}_correct"], results[f"{b}_correct"])]))
    for a, b in pairs
]).round(4)

### Comparison with the earlier runs in `results/`

`rag_pubhealth_n200_3b.csv` was made with `scripts/evaluate.py` and the same 3B model (no CRAG), and
`crag_pubhealth_n200_0.5b.csv` with the 0.5B model. The table also lists the PubHealth accuracy reported in the
CRAG paper (Yan et al., 2024, Table 1), which used LLaMA2 generators and Wikipedia retrieval.

In [ ]:
rows_cmp = []
for path, model_name in [(ROOT / "results" / "crag_pubhealth_n200_0.5b.csv", "Qwen2.5-0.5B (earlier run)"),
                         (ROOT / "results" / "rag_pubhealth_n200_3b.csv", "Qwen2.5-3B (earlier run, no CRAG)")]:
    if path.exists() and path != RESULTS_PATH:
        old = pd.read_csv(path)
        rows_cmp.append({"generator": model_name, "claims": len(old),
                         **{name: (old[col] == old.gold).mean() if col in old else np.nan
                            for col, name in SETTINGS.items()}})
rows_cmp.append({"generator": f"Qwen2.5-{MODEL_KEY.upper()} (this notebook)", "claims": len(results),
                 **summary.accuracy.to_dict()})
rows_cmp += [{"generator": "LLaMA2-hf-7b (CRAG paper)", "claims": np.nan,
              "No retrieval": np.nan, "Standard RAG": 0.489, "CRAG": 0.595},
             {"generator": "SelfRAG-LLaMA2-7b (CRAG paper)", "claims": np.nan,
              "No retrieval": np.nan, "Standard RAG": 0.390, "CRAG": 0.756}]
pd.DataFrame(rows_cmp).set_index("generator").round(3)

In [ ]:
# How closely this run reproduces the earlier 3B run on the settings both have (same claims, same model)
old_path = ROOT / "results" / "rag_pubhealth_n200_3b.csv"
if MODEL_KEY == "3b" and old_path.exists():
    old = pd.read_csv(old_path).drop_duplicates("claim")
    both = results.merge(old, on="claim", suffixes=("", "_earlier"))
    for col in ["no_retrieval", "rag"]:
        same = (both[col] == both[f"{col}_earlier"]).mean()
        print(f"{SETTINGS[col]}: same label as the earlier run for {same:.1%} of {len(both)} claims")

## 7. Example responses

The accuracy runs above skip the explanation. Here a few claims are answered again with `explain=True`, which adds
the evidence sentences, a short reasoning and the response a user would see. The examples are picked to show a claim
that CRAG fixed, one it got wrong, and one that both RAG and CRAG got right.

In [ ]:
picks = []
for mask in [results.crag_correct & ~results.rag_correct,
             ~results.crag_correct & results.rag_correct,
             results.crag_correct & results.rag_correct]:
    if mask.any():
        picks.append(results[mask].iloc[0])

for row in picks[:N_EXAMPLES]:
    print("#" * 100)
    print(f"CLAIM: {row.claim}\nGOLD:  {row.gold}   |   RAG: {row.rag}   |   CRAG: {row.crag} ({row.crag_action})")
    rag_out = run_rag(row.claim, index, model, tokenizer, k=K)
    knowledge, action, _ = correct_knowledge(row.claim, retrieve(index, row.claim, k=K), eval_model,
                                             eval_tokenizer, gen_model=model, gen_tokenizer=tokenizer)
    crag_out = predict_label(model, tokenizer, row.claim, knowledge)
    print(f"\n--- Standard RAG response (P(true) = {rag_out.probs['true']:.2f}) ---\n{rag_out.response}")
    print(f"\n--- CRAG response, action {action} (P(true) = {crag_out.probs['true']:.2f}) ---\n{crag_out.response}\n")

## 8. Save the summary

In [ ]:
summary_path = RESULTS_PATH.with_suffix(".json")
summary_path.write_text(json.dumps({
    "generator": MODELS[MODEL_KEY]["model_name"],
    "device": DEVICE,
    "dtype": str(next(model.parameters()).dtype),
    "claims": len(results),
    "k": K,
    "always_true_accuracy": always_true,
    "settings": summary.round(4).to_dict(orient="index"),
    "by_crag_action": by_action.round(4).to_dict(orient="index"),
}, indent=2))
print("Saved", RESULTS_PATH.relative_to(ROOT), "and", summary_path.relative_to(ROOT))